# Fuse branches → submission.csv  (Track C integration)

Needs `trackc.py` + `b2_foundation.py` next to `data/`. Extracts the **baseline** (spatial, strong on d1) and **B2** (invariant, better on d2/d3) branches over all 6 pools, fuses them with **per-dataset RRF weights**, validates, and writes `submission.csv`.

Per-branch score CSVs are saved (`branch_baseline.csv`, `branch_b2.csv`) — teammates' B1/B3/B4 drop into the same `fuse_branches(...)` call. Extraction is ~10–25 min on GPU (754 volumes); progress prints as it goes.

In [ ]:
import importlib, trackc, b2_foundation as b2
importlib.reload(trackc); importlib.reload(b2)
from trackc import *
import numpy as np, pandas as pd, torch

DATA_ROOT = 'data'; trackc.DATA_ROOT = DATA_ROOT
manifest = load_manifest(DATA_ROOT)
b2.download_weights()
model, device = b2.build_encoder()
print('ready on', device)

## 1. Inline embedders (resample skipped for speed — we resize to a fixed grid anyway)

In [ ]:
@torch.no_grad()
def b2_emb(path):
    vol = load_volume(path, resample_1mm=False, zscore=True, data_root=DATA_ROOT)
    vol = resize_to(vol, b2.INPUT_SIZE)
    x = torch.from_numpy(vol)[None, None].float().to(device)
    out = model.swinViT(x); feat = out[-1] if isinstance(out, (list, tuple)) else out
    v = torch.nn.functional.adaptive_avg_pool3d(feat, 1).flatten().cpu().numpy()
    return (v / (np.linalg.norm(v) + 1e-8)).astype(np.float32)

def base_emb(path):
    return embed_baseline(path, data_root=DATA_ROOT)

def embed_pool(df, id_col, img_col, fn):
    return {r[id_col]: fn(r[img_col]) for _, r in df.iterrows()}

## 2. Extract both branches over all 6 pools → branch score CSVs

In [ ]:
base_rows, b2_rows = [], []
pools = [(k, v) for k, v in manifest.items() if k != 'train_pairs']
for (ds, split), pool in pools:
    for name, fn, bucket in [('baseline', base_emb, base_rows), ('b2', b2_emb, b2_rows)]:
        qE = embed_pool(pool.queries, 'query_id', 'query_image', fn)
        gE = embed_pool(pool.gallery, 'target_id', 'target_image', fn)
        df = embeddings_to_scores_df(qE, gE)
        bucket.append(df)
        print(f'{name:8s} {ds} {split}: {len(qE)}q x {len(gE)}g')

branch_baseline = pd.concat(base_rows, ignore_index=True)
branch_b2 = pd.concat(b2_rows, ignore_index=True)
branch_baseline.to_csv('branch_baseline.csv', index=False)
branch_b2.to_csv('branch_b2.csv', index=False)
print('saved branch_baseline.csv', branch_baseline.shape, '| branch_b2.csv', branch_b2.shape)

## 3. Per-dataset weighted fusion → validated submission.csv
d1: trust the spatial baseline; d2/d3: trust the invariant B2. (Teammates' branches will join this dict.)

In [ ]:
weights = {
    'dataset1': {'baseline': 1.0, 'b2': 0.2},
    'dataset2': {'baseline': 0.3, 'b2': 1.0},
    'dataset3': {'baseline': 0.3, 'b2': 1.0},
}
pool_rankings = fuse_branches({'baseline': branch_baseline, 'b2': branch_b2},
                              manifest, weights_by_dataset=weights)
sub = write_submission(pool_rankings, manifest, path='submission.csv')
validate_submission(sub, manifest)
print('wrote submission.csv'); sub.head(3)

## 4. Local sanity — fusion must not hurt d1 (held-out 50, d1 weights)

In [ ]:
_, hold_df, gt = make_local_split(manifest['train_pairs'], n_holdout=50, seed=0)
qb = embed_pool(hold_df, 'query_id', 'query_image', base_emb)
gb = embed_pool(hold_df, 'target_id', 'target_image', base_emb)
q2 = embed_pool(hold_df, 'query_id', 'query_image', b2_emb)
g2 = embed_pool(hold_df, 'target_id', 'target_image', b2_emb)
r_base = rank_by_embeddings(qb, gb)
r_b2 = rank_by_embeddings(q2, g2)
r_fused = rrf([r_base, r_b2], weights=[1.0, 0.2])  # d1 weights
print('held-out d1 MRR  baseline =', round(mrr(r_base, gt), 4),
      '| b2 =', round(mrr(r_b2, gt), 4),
      '| fused(d1 wts) =', round(mrr(r_fused, gt), 4))

## Done
`submission.csv` is validated (377 rows) — ready to upload to Kaggle for the **first calibration submission** (locks local↔leaderboard agreement).

Paste back: the **`held-out d1 MRR`** line and the **`submission OK`** line. When a teammate hands you a `branch_*.csv` (same `[query_id,target_id,score]` format), add it to the `fuse_branches({...})` dict and re-run cell 3 — no other change.